EXP 16


In [1]:
!pip install -q --only-binary=:all: transformers==5.22.1 sentencepiece gradio

In [2]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def engineering_chatbot(message, history):

    prompt = "Answer the following engineering question in simple English: " + message

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

demo = gr.ChatInterface(
    fn=engineering_chatbot,
    title="Engineering AI Chatbot",
    description="Ask questions about engineering and technology."
)

demo.launch(share=True)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18aa0d2955e839cfc6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


EXP 17

In [3]:
!pip install -q diffusers==0.42.1 "huggingface-hub>=1.32.0,<3.0" accelerate safetensors gradio

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
GPU Available: True
GPU Name: Tesla T4


In [5]:
import torch
import gradio as gr
from diffusers import AutoPipelineForText2Image

model_id = "stabilityai/sd-turbo"

pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")

pipe.enable_attention_slicing()

def generate_image(prompt):

    if not prompt.strip():
        raise gr.Error("Please enter an image description.")

    image = pipe(
        prompt=prompt,
        num_inference_steps=1,
        guidance_scale=0.0,
        height=512,
        width=512
    ).images[0]

    return image

demo = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(
        label="Enter Your Text Prompt",
        placeholder="A beautiful futuristic engineering laboratory",
        lines=3
    ),
    outputs=gr.Image(type="pil", label="Generated Image"),
    title="AI Text-to-Image Generator",
    description="Generate AI images from text descriptions using Stable Diffusion Turbo."
)

demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd31188e9ceab94dc1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


EXP 18

In [6]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q openai-whisper gTTS gradio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.22.1 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 1.16.1 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [7]:
import whisper
import gradio
import torch
from gtts import gTTS

print("Whisper:", whisper.__file__)
print("Gradio:", gradio.__version__)
print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
print("All required libraries imported successfully!")

Whisper: /usr/local/lib/python3.13/dist-packages/whisper/__init__.py
Gradio: 6.31.0
PyTorch: 2.14.1+cu130
GPU Available: True
All required libraries imported successfully!


In [8]:
import whisper
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = whisper.load_model("base", device=device)

print("Whisper model loaded successfully!")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 91.5MiB/s]


Whisper model loaded successfully!


In [9]:
import gradio as gr
from gtts import gTTS
import tempfile

def speech_to_text(audio):

    if audio is None:
        return "Please upload or record audio."

    result = model.transcribe(audio)

    return result["text"]


def text_to_speech(text, language):

    if not text.strip():
        raise gr.Error("Please enter some text.")

    filename = tempfile.NamedTemporaryFile(
        suffix=".mp3",
        delete=False
    ).name

    tts = gTTS(text=text, lang=language)
    tts.save(filename)

    return filename


with gr.Blocks() as demo:

    gr.Markdown("# AI Speech-to-Text and Text-to-Speech")

    with gr.Tab("Speech to Text"):

        audio_input = gr.Audio(
            sources=["upload", "microphone"],
            type="filepath",
            label="Upload or Record Your Voice"
        )

        text_output = gr.Textbox(
            label="Recognized Text",
            lines=5
        )

        convert1 = gr.Button("Convert Speech to Text")

        convert1.click(
            speech_to_text,
            inputs=audio_input,
            outputs=text_output
        )

    with gr.Tab("Text to Speech"):

        text_input = gr.Textbox(
            label="Enter Your Text",
            lines=5
        )

        language = gr.Dropdown(
            choices=[
                ("English", "en"),
                ("Tamil", "ta"),
                ("Telugu", "te"),
                ("Hindi", "hi")
            ],
            value="en",
            label="Select Language"
        )

        audio_output = gr.Audio(
            label="Generated Speech",
            type="filepath"
        )

        convert2 = gr.Button("Convert Text to Speech")

        convert2.click(
            text_to_speech,
            inputs=[text_input, language],
            outputs=audio_output
        )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9893a81bf0d6ec90b1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


EXP 19

In [11]:
import transformers
import gradio
import torch

print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)
print("PyTorch:", torch.__version__)

Transformers: 5.22.1
Gradio: 6.31.0
PyTorch: 2.14.1+cu130


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading summarization model...")

summary_name = "sshleifer/distilbart-cnn-12-6"

summary_tokenizer = AutoTokenizer.from_pretrained(summary_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    summary_name
).to(device)

print("Loading translation model...")

translation_name = "Helsinki-NLP/opus-mt-en-fr"

translation_tokenizer = AutoTokenizer.from_pretrained(translation_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_name
).to(device)

print("Both models loaded successfully!")
print("Device:", device)

Loading summarization model...


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

... (15 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  301MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Both models loaded successfully!
Device: cuda


In [13]:
import gradio as gr
import torch

def summarize_text(text):

    if not text.strip():
        return "Please enter a document."

    inputs = summary_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = summary_model.generate(
            **inputs,
            max_new_tokens=120,
            min_new_tokens=20,
            num_beams=4
        )

    return summary_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


def translate_text(text):

    if not text.strip():
        return "Please enter English text."

    inputs = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=500
    ).to(device)

    with torch.no_grad():
        outputs = translation_model.generate(
            **inputs,
            max_new_tokens=200,
            num_beams=4
        )

    return translation_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


with gr.Blocks() as demo:

    gr.Markdown("# AI Document Summarization and Translation")

    with gr.Tab("Document Summarization"):

        document = gr.Textbox(
            label="Enter Document",
            lines=8
        )

        summary = gr.Textbox(
            label="Generated Summary",
            lines=5
        )

        gr.Button("Summarize Document").click(
            summarize_text,
            inputs=document,
            outputs=summary
        )

    with gr.Tab("Machine Translation"):

        original = gr.Textbox(
            label="Enter English Text",
            lines=5
        )

        translated = gr.Textbox(
            label="French Translation",
            lines=5
        )

        gr.Button("Translate Text").click(
            translate_text,
            inputs=original,
            outputs=translated
        )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25df8077a14dac816a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


EXP 20

In [14]:
!pip install -q sentence-transformers gradio

In [15]:
import torch
from sentence_transformers import SentenceTransformer, util

print("Loading AI model...")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("AI model loaded successfully!")

if torch.cuda.is_available():
    print("Running on GPU")
else:
    print("Running on CPU")

Loading AI model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AI model loaded successfully!
Running on GPU


In [16]:
import gradio as gr
import torch
from sentence_transformers import util

def screen_resume(resume, job_description):

    if not resume.strip() or not job_description.strip():
        raise gr.Error("Please enter both resume and job description.")

    resume_embedding = model.encode(resume, convert_to_tensor=True)
    job_embedding = model.encode(job_description, convert_to_tensor=True)

    similarity = util.cos_sim(resume_embedding, job_embedding).item()

    score = round(similarity * 100, 2)

    if score >= 70:
        result = "Strong Match"
    elif score >= 40:
        result = "Moderate Match"
    else:
        result = "Low Match"

    return f"Resume Similarity Score: {score}%\nResult: {result}"

with gr.Blocks() as demo:

    gr.Markdown("# AI-Based Resume Screening System")
    gr.Markdown("Compare a candidate's resume with a job description.")

    resume = gr.Textbox(
        label="Enter Candidate Resume",
        lines=10,
        placeholder="Paste the candidate's resume here..."
    )

    job = gr.Textbox(
        label="Enter Job Description",
        lines=8,
        placeholder="Paste the job requirements here..."
    )

    button = gr.Button("Screen Resume")

    output = gr.Textbox(
        label="Screening Result",
        lines=3
    )

    button.click(
        fn=screen_resume,
        inputs=[resume, job],
        outputs=output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1928111092a1437527.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
